# Transferred chunk — surface maps

Loads the **transferred chunk** back from S3 (the data written by the chunk
transfer, now living in local NRP S3) and plots the **surface layer** of every
variable in the config's `variables` list (except `time`).

* Run on the MIT server (or anywhere with NRP S3 credentials configured).
* Reads only one timestamp's chunk store + the chunk `grid.zarr`.
* No quantities are computed — just the surface slice (`k=0`) per variable.
* Reuses repo code:
  * read S3: `get_raw_data.get_llc_timestep_data`, `get_raw_data.get_llc_depth_gridfile`
  * surface slice: `depth_strategies.select_surface` (+ `vertical_helpers._get_vertical_dim`)
  * colormaps/labels: `plotting.field_cmaps.load_field_cmaps`
  * map drawing: `plotting.global_maps.plot_global_field`
* Requires **cmocean** and **cartopy**
  (`conda install -c conda-forge cmocean cartopy`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs


def _find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError(f"Could not locate repo root (pyproject.toml) above {p}")


REPO = _find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dbof.transfer import config as tconfig
from dbof.llc4320_ingestion import get_raw_data
from dbof.preprocessing.depth_strategies import select_surface
from dbof.preprocessing.vertical_helpers import _get_vertical_dim
from dbof.plotting.field_cmaps import load_field_cmaps
from dbof.plotting import global_maps

print("repo root:", REPO)

In [ ]:
# --- Config -> S3 location of the transferred chunk ---
CONFIG = REPO / "configs/transfer/run_chunks_monterey_bay.yaml"
cfg = tconfig.load_config(str(CONFIG))
loc = cfg.transfer.location

# Folder layout written by the transfer: {raw_prefix}/{chunks_subdir}/{chunk_name}
folder = f"{cfg.output.raw_prefix}/{cfg.output.chunks_subdir}/{loc.chunk_name}"
endpoint = cfg.output.s3_endpoint
bucket = cfg.output.bucket

# Pick the first transferred date (edit to inspect another).
DATE = cfg.data.date_iterations[0]
plot_vars = [v for v in cfg.transfer.variables if v != "time"]
print(f"reading chunk '{loc.chunk_name}' from s3://{bucket.strip('/')}/{folder}")
print(f"date = {DATE}")
print(f"variables = {plot_vars}")

In [ ]:
# --- Read the chunk grid (XC/YC) and the timestep fields from S3 ---
grid = get_raw_data.get_llc_depth_gridfile(endpoint, bucket, folder)
XC = np.asarray(grid["XC"].squeeze().values)
YC = np.asarray(grid["YC"].squeeze().values)

ds = get_raw_data.get_llc_timestep_data(
    endpoint, bucket, folder, DATE,
    vars_requested=plot_vars,
    chunks=get_raw_data.llc_depth_timestep_chunks,
    storage_options=get_raw_data._llc_depth_storage_options(endpoint),
)
print("loaded vars:", list(ds.data_vars))
print("dims:", dict(ds.sizes))

In [ ]:
cmaps, diverging_cmaps = load_field_cmaps()


def surface_slice(da):
    """Return the k=0 surface slice for a 3D field; pass 2D fields through.

    Uses the repo's select_surface (which picks k=0 / k_l=0) when a vertical
    dimension is present.
    """
    try:
        _get_vertical_dim(da)
    except ValueError:
        return da              # already 2D (Eta, oceTAUX, ... )
    return select_surface(da, ds)

In [ ]:
proj = ccrs.PlateCarree()
ncol = 4
nrow = int(np.ceil(len(plot_vars) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.2 * nrow),
                         subplot_kw={"projection": proj})
axes = np.atleast_1d(axes).ravel()

extent = [float(np.nanmin(XC)), float(np.nanmax(XC)),
          float(np.nanmin(YC)), float(np.nanmax(YC))]

for ax, var in zip(axes, plot_vars):
    arr = np.asarray(surface_slice(ds[var]).squeeze().values)
    im, label = global_maps.plot_global_field(
        ax, XC, YC, arr, var, cmaps,
        diverging_cmaps=diverging_cmaps,
        transform=proj, add_coastline=True,
    )
    ax.set_extent(extent, crs=proj)
    ax.set_title(var)
    if im is not None:
        cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03)
        cb.set_label(label, fontsize=8)
    else:
        ax.text(0.5, 0.5, f"{var}\n(all NaN)", ha="center", va="center",
                transform=ax.transAxes)

# Hide any unused panels.
for ax in axes[len(plot_vars):]:
    ax.set_visible(False)

fig.suptitle(f"{loc.chunk_name} surface fields — {DATE}", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()